In [5]:
# ======================================================
# SECTION 1 — IMPORTS
# ======================================================

import os
import re
import glob
import h5py

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

# optional display settings
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 0)

In [12]:
# ======================================================
# SECTION 2 — PATHS
# ======================================================

# ======================================================
# COHORT 1
# ======================================================

cohort1_h5_dir = r"C:\Users\sjs93\UFL Dropbox\Sequioa Smith\Padilla-Coreano Lab\2025\ECG_cohort1\Aim1\AIM1\Day1_new\CM interactions\cm_h5\baseline_cagemate_interactions_h5_outputs"

cohort1_boris_dir = r"C:\Users\sjs93\UFL Dropbox\Sequioa Smith\Padilla-Coreano Lab\2025\ECG_cohort1\Aim1\AIM1\Day1_new\CM interactions\cm_boris\baseline csv"


# ======================================================
# COHORT 2
# ======================================================

cohort2_h5_dir = r"C:\Users\sjs93\UFL Dropbox\Sequioa Smith\Padilla-Coreano Lab\2025\BLA_ChR_resp_pilot1 (SS)\BLA_resp_ChR_cm_hc\h5_outputs"

cohort2_boris_dir = r"C:\Users\sjs93\UFL Dropbox\Sequioa Smith\Padilla-Coreano Lab\2025\BLA_ChR_resp_pilot1 (SS)\BLA_resp_ChR_cm_boris"


In [13]:
# ======================================================
# SECTION 3 — BUILD COHORT 1 METADATA
# ======================================================

cohort1_h5_files = glob.glob(
    os.path.join(cohort1_h5_dir, "*.h5")
)

rows = []

for path in cohort1_h5_files:

    filename = os.path.basename(path)

    stem = filename.replace(".h5", "")

    print(stem)

    # --------------------------------------------------
    # parse filename
    # --------------------------------------------------

    match = re.search(
        r'CM_s(\d+)_(\d+)_(d|sub)(\d+)_(\d+)',
        stem
    )

    if match is None:

        print("FAILED:", filename)
        continue

    # --------------------------------------------------
    # IDs
    # --------------------------------------------------

    subject_id = (
        f"{match.group(1)}_{match.group(2)}"
    )

    social_agent_id = (
        f"{match.group(4)}_{match.group(5)}"
    )

    # --------------------------------------------------
    # cohort
    # --------------------------------------------------

    cohort = "1"

    # --------------------------------------------------
    # ranks
    # --------------------------------------------------

    relation = match.group(3)

    if relation == "d":

        subject_rank = "DOM"
        social_agent_rank = "SUB"

        relative_interaction = "DOM"

    elif relation == "sub":

        subject_rank = "SUB"
        social_agent_rank = "DOM"

        relative_interaction = "SUB"

    else:

        subject_rank = "UNKNOWN"
        social_agent_rank = "UNKNOWN"

        relative_interaction = "UNKNOWN"

    # --------------------------------------------------
    # session key
    # --------------------------------------------------

    session_match = re.search(
        r'(\d{8}_\d{6})',
        stem
    )

    session_key = (
        session_match.group(1)
        if session_match
        else "UNKNOWN"
    )

    # --------------------------------------------------
    # row
    # --------------------------------------------------

    row = {

        "SessionKey":
            session_key,

        "Filename":
            filename,

        "H5Path":
            path,

        "SubjectID":
            subject_id,

        "SocialAgentID":
            social_agent_id,

        "Cohort":
            cohort,

        "SubjectRank":
            subject_rank,

        "SocialAgentRank":
            social_agent_rank,

        "RelativeInteraction":
            relative_interaction
    }

    rows.append(row)

cohort1_meta = pd.DataFrame(rows)

print(cohort1_meta.shape)

display(cohort1_meta.head())

CM_s1_1_d1_2_20250623_111352_merged
CM_s1_2_sub1_1_20250623_133932_merged
CM_s2_3_d2_4_20250623_151153_merged
CM_s2_4_sub2_3_20250623_143348_merged
CM_s3_5_d3_6_20250623_160001_merged
CM_s3_5_d3_6_20250623_170708_merged
CM_s3_6_sub3_5_20250623_174348_merged
CM_s4_7_d4_8_20250623_193718_merged
CM_s4_8_sub4_7_20250623_182649_merged
(9, 9)


,SessionKey,Filename,H5Path,SubjectID,SocialAgentID,Cohort,SubjectRank,SocialAgentRank,RelativeInteraction
0,20250623_111352,CM_s1_1_d1_2_20250623_111352_merged.h5,C:\Users\sjs93\UFL Dropbox\Sequioa Smith\Padil...,1_1,1_2,1,DOM,SUB,DOM
1,20250623_133932,CM_s1_2_sub1_1_20250623_133932_merged.h5,C:\Users\sjs93\UFL Dropbox\Sequioa Smith\Padil...,1_2,1_1,1,SUB,DOM,SUB
2,20250623_151153,CM_s2_3_d2_4_20250623_151153_merged.h5,C:\Users\sjs93\UFL Dropbox\Sequioa Smith\Padil...,2_3,2_4,1,DOM,SUB,DOM
3,20250623_143348,CM_s2_4_sub2_3_20250623_143348_merged.h5,C:\Users\sjs93\UFL Dropbox\Sequioa Smith\Padil...,2_4,2_3,1,SUB,DOM,SUB
4,20250623_160001,CM_s3_5_d3_6_20250623_160001_merged.h5,C:\Users\sjs93\UFL Dropbox\Sequioa Smith\Padil...,3_5,3_6,1,DOM,SUB,DOM


In [18]:
# ======================================================
# SECTION 4 — BUILD COHORT 2 METADATA
# ======================================================

cohort2_boris_files = glob.glob(
    os.path.join(cohort2_boris_dir, "*.csv")
)

rank_map = {
    "d": "DOM",
    "i": "INT",
    "s": "SUB"
}

rank_order = {
    "SUB": 0,
    "INT": 1,
    "DOM": 2
}

rows = []

for path in cohort2_boris_files:

    filename = os.path.basename(path)

    print(filename)

    # --------------------------------------------------
    # FIX malformed filename
    # --------------------------------------------------

    fixed_filename = filename

    if "_s_1_2_" in fixed_filename:

        fixed_filename = fixed_filename.replace(
            "_s_1_2_",
            "_s_cm_1_2_"
        )

        print("FIXED:", fixed_filename)

    # --------------------------------------------------
    # parse filename
    # --------------------------------------------------

    match = re.search(
        r'(\d+)_(\d+)_(\d+)_([dis])_cm_(\d+)_(\d+)_([dis])',
        fixed_filename
    )

    if match is None:

        print("FAILED:", filename)
        continue

    # --------------------------------------------------
    # IDs
    # --------------------------------------------------

    subject_id = (
        f"{match.group(1)}_{match.group(2)}"
    )

    social_agent_id = (
        f"{match.group(5)}_{match.group(6)}"
    )

    # --------------------------------------------------
    # cohort
    # --------------------------------------------------

    cohort = match.group(3)

    # --------------------------------------------------
    # ranks
    # --------------------------------------------------

    subject_rank = rank_map[
        match.group(4)
    ]

    social_agent_rank = rank_map[
        match.group(7)
    ]

    # --------------------------------------------------
    # relative interaction
    # --------------------------------------------------

    if rank_order[subject_rank] > rank_order[social_agent_rank]:

        relative_interaction = "DOM"

    elif rank_order[subject_rank] < rank_order[social_agent_rank]:

        relative_interaction = "SUB"

    else:

        relative_interaction = "EQUAL"

    # --------------------------------------------------
    # session key
    # --------------------------------------------------

    session_match = re.search(
        r'(\d{8}_\d{6})',
        fixed_filename
    )

    session_key = (
        session_match.group(1)
        if session_match
        else "UNKNOWN"
    )

    # --------------------------------------------------
    # row
    # --------------------------------------------------

    row = {

        "SessionKey":
            session_key,

        "Filename":
            filename,

        "BORISPath":
            path,

        "SubjectID":
            subject_id,

        "SocialAgentID":
            social_agent_id,

        "Cohort":
            cohort,

        "SubjectRank":
            subject_rank,

        "SocialAgentRank":
            social_agent_rank,

        "RelativeInteraction":
            relative_interaction
    }

    rows.append(row)

# ======================================================
# DATAFRAME
# ======================================================

cohort2_meta = pd.DataFrame(rows)

print(cohort2_meta.shape)

display(cohort2_meta.head())

1_1_2_i_cm_1_2_d_20260106_134350.1_BORIS_SS.csv
1_1_2_i_cm_1_3_s_20260107_163611.1_VT_CM.csv
1_2_2_d_cm_1_3_s_20260108_114416.1_VT_CM.csv
1_3_2_s_1_2_d_20260107_165340.1_boris_AJ_with_image_indices_with_image_indices.csv
FIXED: 1_3_2_s_cm_1_2_d_20260107_165340.1_boris_AJ_with_image_indices_with_image_indices.csv
1_3_2_s_cm_1_1_i_20260106_122823.1_boris_AJ_with_image_indices.csv
2_1_2_d_cm_2_2_s_20260108_112426.1_VT_CM.csv
2_1_2_d_cm_2_3_i_20260106_105109_BORIS_SS.csv
2_2_2_s_cm_2_1_d_20260108_120453.1_VT_CM.csv
2_2_2_s_cm_2_3_i_20260106_132549.1_boris_AJ_with_image_indices.csv
3_1_2_i_cm_3_2_d_20260106_115157.1_BORIS_SS.csv
3_1_2_i_cm_3_3_s_20260107_123230.1_BORIS_SS.csv
3_2_2_d_cm_3_3_s_20260107_140924.1_boris_AJ_with_image_indices.csv
3_3_2_s_cm_3_1_i_20260106_124546.1_boris_AJ_with_image_indices.csv
3_3_2_s_cm_3_2_d_20260107_171330.1_VT_CM.csv
4_1_2_s_cm_4_2_d_20260107_160439.1_VT_CM.csv
4_2_2_d_cm_4_1_s_20260107_150836.1_VT_CM.csv
4_3_2_d_cm_4_1_s_20260107_113120.1_boris_AJ_with_im

,SessionKey,Filename,BORISPath,SubjectID,SocialAgentID,Cohort,SubjectRank,SocialAgentRank,RelativeInteraction
0,20260106_134350,1_1_2_i_cm_1_2_d_20260106_134350.1_BORIS_SS.csv,C:\Users\sjs93\UFL Dropbox\Sequioa Smith\Padil...,1_1,1_2,2,INT,DOM,SUB
1,20260107_163611,1_1_2_i_cm_1_3_s_20260107_163611.1_VT_CM.csv,C:\Users\sjs93\UFL Dropbox\Sequioa Smith\Padil...,1_1,1_3,2,INT,SUB,DOM
2,20260108_114416,1_2_2_d_cm_1_3_s_20260108_114416.1_VT_CM.csv,C:\Users\sjs93\UFL Dropbox\Sequioa Smith\Padil...,1_2,1_3,2,DOM,SUB,DOM
3,20260107_165340,1_3_2_s_1_2_d_20260107_165340.1_boris_AJ_with_...,C:\Users\sjs93\UFL Dropbox\Sequioa Smith\Padil...,1_3,1_2,2,SUB,DOM,SUB
4,20260106_122823,1_3_2_s_cm_1_1_i_20260106_122823.1_boris_AJ_wi...,C:\Users\sjs93\UFL Dropbox\Sequioa Smith\Padil...,1_3,1_1,2,SUB,INT,SUB
